<div align="center">
  <hr>
  <p text-align='center'><strong>Guía para la integración de algoritmos de aprendizaje automático en EmbedIA.</strong></p>
  <img src="https://raw.githubusercontent.com/Embed-ML/EmbedIA/main/docs/assets/images/logo3.png" width=20%/>
  <hr>
</div>

Antes de comenzar esta guía repasemos brevemente y de manera simplificada el flujo de trabajo que, llegando al final de la misma, deberemos seguir:
* El usuario va a entrenar un modelo de aprendizaje automático utilizando la conocida biblioteca scikit-learn, el cual deberá guardar.
* Ese modelo con los parámetros de entrenamiento guardados será cargado y luego convertido por EmbedIA para poder realizar inferencias en microcontroladores.
<br>
<div align="center">
<img src="https://github.com/Embed-ML/EmbedIA/raw/main/docs/assets/images/workflow.png" width=60%/>
</div>

Pero...¿Cómo hace EmbedIA para convertir un modelo entrenado en scikit-learn a una implementación en C que pueda funcionar en un microcontrolador?. Para responder esta pregunta iremos paso a paso por cada una de las secciones de la <a href='#tabla-de-contenidos'>tabla de contenidos</a>. A lo largo del documento, se utilizará <strong>a modo de ejemplo</strong>, la integración del algoritmo KNeighborsClassifier (una implementación específica de K-nearest neighbors (KNN) que se utiliza para problemas de clasificación).

## Tabla de contenidos <A NAME="tabla-de-contenidos"></A>
1. [Investigación](#investigation)
2. [Desarrollo del código en C](#coding)
  - [Generación de la función init](#generation)
3. [Integrando el código en EmbedIA](#integration)
  - [Carpeta Libraries](#libraries)
  - [Carpeta Core](#core)
  - [Carpeta Wrappers](#wrappers)
  - [Carpeta Layers](#layers)
  - [Registro en layers_implemented.py](#registration)
4. [Usando EmbedIA](#usingembedia)
  - [Posibles errores](#errors)


## Investigación 🔍 <A NAME="investigation"></A>
El primer paso es investigar un algoritmo de aprendizaje automatico y entender su funcionamiento. Entrando en la página de <a href="https://scikit-learn.org/stable/index.html">scikit-learn</a>, pueden observarse varias categorías dependiendo el tipo de problema que se quiere resolver. Al elegir uno de estos, vamos a ver en la página correspondiente los parámetros de entrenamiento que posee (que es lo que más nos va a interesar en un principio), atributos y algunos ejemplos. En particular, para el algoritmo <a href="https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html#sklearn.neighbors.KNeighborsClassifier">KNeighborsClassifier</a> podemos ver que algunos de sus parámetros son ```n_neighbors```, ```weights```, entre otros.<br>
Para mejor comprensión, también es de utilidad buscar en blogs o videos, explicaciones teóricas para entender las partes más importantes del algoritmo y tener una visión más amplia del mismo.


## Desarrollo del código en C 👨‍💻 <A NAME="coding"></A>
Una vez entendidos los conceptos que rodean al algoritmo deseado, es hora de implementarlo en C. Para esto va a ser necesario crear tres archivos:
* <strong>'[nombre del algoritmo].h': </strong>Se declara la estructura del algoritmo, es decir, los parámetros de entrenamiento y otras variables que el desarrollador considere necesarios.
```c
// Ejemplo de estructura de KNeighborsClassifier en 'knn.h',
// en la que se añade únicamente el parámetro n_neighbors
typedef struct
{
    int n_neighbors;
    int n_rows;
    int n_features;
    float *neighbors_features;
    float *neighbors_labels;
} KNN;
// Cabecera de la función de predicción
int predict(KNN, float*);
```

* <strong>'[nombre del algoritmo].c': </strong>En este archivo se declaran dos funciones principales: La de predicción y la de inicialización. La primera recibe por parámetros la estructura creada anteriormente y un elemento del que se quiere inferir el resultado. <a href='#generation'>La segunda</a>, por su parte y como su nombre lo dice, inicializa todas las variables de la estrutura y retorna esta última.<br>
Además de estas dos, en este archivo se agregan todas las funciones auxiliares necesarias que va a utilizar la función de predicción.
* <strong>'main.c': </strong>Contiene al programa principal. Se crea la estructura llamando a la funcion de inicialización y luego se realiza la inferencia utilizando la funcion creada para predecir.

### Generación de la función init <A NAME="generation"></A>
Para acercarnos un poco más al funcionamiento de EmbedIA y para poder probar el código desarrollado en el paso anterior, vamos a generar el código de la función de inicialización mediante python. Para esto, vamos a crear un string al que le aplicaremos la función format(), donde cada parámetro que se le pase a esta última será reemplazado en el texto. Dentro del string, cada parámetro a reemplazar debe llevar llaves ({n_neighbors}), y como C usa llaves para definir inicios y finales, estas deben duplicarse para indicar que lo que sigue no es un parámetro.
<br>
Siguiendo con el ejemplo de KNeighborsClassifier, el string sería el siguiente:


In [ ]:
code_str = """
KNN init() {{
    static float neighbors_features[][{n_features}] = {{ {features_data} }};
    static int neighbors_labels[] = {{ {labels_data} }};

    KNN knn = {{ {n_neighbors}, {n_rows}, {n_features}, *neighbors_features, neighbors_labels }};
    return knn;
}}
"""
x_train = [1, 0.5, 3]
y_train = [0, 0, 1]

ready_code = code_str.format(
    n_neighbors=3,
    n_rows=len(y_train),
    n_features=1,
    features_data=','.join([str(x) for x in x_train]),
    labels_data=','.join([str(y) for y in y_train])
)

print(ready_code)


KNN init() {
    static float neighbors_features[][1] = { 1,0.5,3 };
    static int neighbors_labels[] = { 0,0,1 };

    KNN knn = { 3, 3, 1, *neighbors_features, neighbors_labels };
    return knn;
}



Quizá al utilizar un dataset real, copiar el código generado se vuelva engorroso, por lo que puede añadirse todo lo desarrollado en el archivo [nombre del algoritmo].c dentro del string, formatearlo al igual que la función de inicialización y luego descargarlo utilizando las siguientes líneas:
```python
with open('[nombre del algoritmo].c', 'w') as file:
  file.write(ready_code)
```
Con estas nociones y el algoritmo funcionando en C, ya podemos empezar a tocar la libería de EmbedIA 🥳

## Integración del código en EmbedIA⚙️ <A NAME="integration"></A>
Las carpetas que vamos a utilizar principalmente son: core, layers, libraries y wrappers

### Carpeta Libraries 📁 <A NAME="libraries"></A>
Dentro de esta carpeta se encuentran los algoritmos de aprendizaje automático y aprendizaje automático profundo. La estructura está organizada por microcontrolador/plataforma:

- **📁 `mcu/generic/`**: Implementaciones genéricas que funcionan en cualquier plataforma. Dentro encontrarás carpetas agrupadas según el tipo de dato de entrada y salida (```float```, ```fixed16```, ```fixed32```, ```quant8```, etc.). Por ejemplo, en 📁`generic/float` encontrarás:
  - ```neural_net.c``` y ```neural_net.h```: Código de algoritmos de redes neuronales.
  - ```common.h``` y ```common.c```: Estructura de los datos de entrada y salida, como **data1d_t** para datos de una dimensión tipo flotante.
  - Otros archivos de algoritmos específicos (```knn.c```, ```knn.h```, etc.)

- **📁 `mcu/esp32/`**, **📁 `mcu/[otra plataforma]/`**, etc.: Implementaciones optimizadas específicas para cada microcontrolador. Siguen la misma estructura que `generic` pero con código optimizado para esa plataforma.

- **📁 `template/`**: Declaraciones comunes y referencias que pueden ser incluidas desde implementaciones específicas. Si un archivo no existe en una carpeta de plataforma específica (ej: `esp32/float/`), el sistema buscará en `generic` automáticamente.

<strong>Agregando tu algoritmo:</strong>
No debes tocar los archivos existentes. En su lugar, debes crear dos nuevos archivos: **[nombre del algoritmo].c** y **[nombre del algoritmo].h** que habías desarrollado en la sección anterior. Estos archivos se agregarán a la carpeta correspondiente (por ejemplo, `libraries/mcu/generic/float/`), con los siguientes cambios convencionales en EmbedIA:

- La función de predicción debe tener '_layer' al final de su nombre. Ej: 'knn_layer'. No devuelve nada, sus parámetros son la estructura, los datos de entrada y de salida (donde se almacena el resultado).
- El nombre de la estructura debe terminar con '_layer_t'.
- Los tipos de datos deben coincidir con los definidos en 'common.h' (pasar `int` a `uint16_t`, `float` a `float`, etc.).

Siguiendo con el ejemplo de KNeighborsClassifier, el nuevo 'knn.h' quedaría así:

```c
// Estructura del algoritmo
typedef struct
{
    uint16_t n_neighbors;
    uint32_t n_rows;
    uint16_t n_features;
    float *neighbors_features;
    float *neighbors_labels;
}k_neighbors_classifier_layer_t;
// Cabecera de la función de predicción
void k_neighbors_classifier_layer(k_neighbors_classifier_layer_t layer, data1d_t input, data1d_t * output);
```
### Carpeta Core 📁 <A NAME="core"></A>
En esta carpeta residen las clases base que definen el comportamiento general de cada tipo de algoritmo. Aquí es donde debes crear un archivo llamado **'[nombre del algoritmo]_base_layer.py'** que define la clase **[nombre del algoritmo]BaseLayer**, similar a `NeuralNetLayer` en `neural_net_layer.py`.

Esta clase base:
- Hereda de `Layer`
- Define los archivos C requieridos mediante la propiedad `required_files`
- Implementa métodos para calcular parámetros, operaciones MAC y memoria

<strong>La clase `EmbediaFile`:</strong>
`EmbediaFile` es una clase que representa un archivo requerido con opciones de inyección de defines en el código C generado. Esto permite que el generador de código sepa:
1. **Qué archivo incluir** (nombre del .h o .c)
2. **Desde dónde buscarlo** (generic → esp32 → template, según disponibilidad)
3. **Qué defines agregar** en el archivo para configurar comportamientos y/o valores específicos

```python
class EmbediaFile:
    def __init__(self, name, defines=None):
        self.name = name           # Nombre del archivo: "knn.h"
        self.defines = defines     # Defines opcionales: {"CONSTANTE SIMBOLICA": valor}
```

<strong>Cómo funciona el sistema de búsqueda de archivos:</strong>
Cuando generas un proyecto para `esp32/float`, EmbedIA busca archivos en este orden:
```
1. libraries/mcu/template/archivo.h         ← Primero (plantilla base)
2. libraries/mcu/esp32/float/archivo.h      ← Segundo (especifico)
3. libraries/mcu/generic/float/archivo.h    ← Tercero (genérico, fallback)
```

Esto permite tener implementaciones optimizadas por plataforma sin duplicar código.

<strong>Ejemplo de `required_files` en knn_base_layer.py:</strong>

```python
from embedia.core.layer import Layer, EmbediaFile
from embedia.model_generator.project_options import ModelDataType
from math import log2

class KnnBaseLayer(Layer):
    def __init__(self, model, wrapper, **kwargs):
        super().__init__(model, wrapper, **kwargs)

    @property
    def required_files(self):
        '''
        Retorna una lista de tuplas indicando los archivos (.h) y (.c) requeridos.
        EmbediaFile permite especificar defines opcionales.
        '''
        return super().required_files + [
            # Busca knn.h/c en: esp32/float → generic/float → template/
            (EmbediaFile('knn.h'), EmbediaFile('knn.c')),

            # Busca distances.h/c, pero la implementación esp32/float puede tener
            # optimizaciones específicas que la genérica no tiene
            (EmbediaFile('distances.h'), EmbediaFile('distances.c')),

            # Ejemplo con defines: busca signals.h pero le inyecta ENABLE_FFT_OPTIMIZATIONS
            # (EmbediaFile('signals.h', defines={'SAME_VALUE': 5}),
            #  EmbediaFile('signals.c'))
        ]

    def calculate_params(self):
        # ... implementación para calcular parámetros
        pass

    def calculate_MAC(self):
        # ... implementación para calcular operaciones MAC
        pass

    def calculate_memory(self):
        # ... implementación para calcular memoria
        pass
```

<strong>¿Cuándo usar defines en EmbediaFile?</strong>
- **Sin defines**: Archivo estándar que funciona igual en todas las implementaciones
- **Con defines**: El archivo tiene código condicional (#ifdef) que se activa solo para cierta plataforma o configuración

Ejemplo en un archivo C:
```c
// En libraries/mcu/distances.c
void calculate_distance(float *a, float *b, float *result) {
    #ifdef KNN_OPTIMIZED_ESP32
        // Implementación optimizada para ESP32
        asm volatile("...optimizaciones específicas...");
    #else
        // Implementación genérica
        // ... código estándar ...
    #endif
}
```


### Carpeta Wrappers 📁 <A NAME="wrappers"></A>
La arquitectura de wrappers define las **interfaces que cada algoritmo necesita** y proporciona **implementaciones específicas para cada librería**.

<strong>Concepto clave: Interfaz Base vs Implementación</strong>
- **Interfaz Base** (raíz de wrappers/): Define QUÉ propiedades/métodos necesita un algoritmo para ser integrado en EmbedIA. Es independiente de la librería.
- **Implementación** (carpetas por librería): Implementa CÓMO extraer esas propiedades de un modelo específico de una librería.

<strong>Estructura actual:</strong>
```
wrappers/
├── layer_wrapper.py              ← Base abstracta de todos los wrappers
├── embedia_wrappers.py           ← Interfaz para modelos EmbedIA
├── neural_net_base.py            ← Interfaz para redes neuronales
├── tree_base.py                  ← Interfaz para algoritmos de árbol
├── svm_base.py                   ← Interfaz para SVM
├── distance_base.py              ← Interfaz para algoritmos basados en distancia (KNN, etc.)
│
├── sklearn/                       ← Implementaciones para scikit-learn
│   └── sklearn_wrappers.py       ← SKLKnnWrapper, SKLSVMWrapper, etc.
├── tensorflow/                    ← Implementaciones para TensorFlow/Keras
│   └── tensorflow_wrappers.py    ← TFNeuralNetWrapper, etc.
└── larq/                          ← Implementaciones para Larq
    └── larq_wrappers.py          ← LarqBinaryDenseWrapper, etc.
```

<strong>Definir una interfaz base (Ejemplo: distance_base.py):</strong>
Aquí defines qué propiedades NECESITA cualquier algoritmo basado en distancia (como KNN):

```python
class DistanceBaseWrapper(LayerWrapper):
    """Define la interfaz que necesita un algoritmo basado en distancia."""

    @property
    def n_neighbors(self) -> int:
        """Número de vecinos a considerar"""
        raise NotImplementedError

    @property
    def n_samples(self) -> int:
        """Número de muestras de entrenamiento"""
        raise NotImplementedError

    @property
    def n_features(self) -> int:
        """Número de características"""
        raise NotImplementedError

    @property
    def input_shape(self) -> tuple:
        """Forma del input"""
        raise NotImplementedError

    @property
    def output_shape(self) -> tuple:
        """Forma del output"""
        raise NotImplementedError
```

<strong>Implementar para scikit-learn (Ejemplo: sklearn_wrappers.py):</strong>
Aquí implementas CÓMO extraer esas propiedades de un modelo de sklearn:

```python
class SKLKnnWrapper(DistanceBaseWrapper, ScikitLearnWrapper):
    """Implementa la interfaz DistanceBaseWrapper para sklearn KNeighborsClassifier."""

    @property
    def n_neighbors(self):
        return self._target.n_neighbors  # Extraído del modelo sklearn

    @property
    def n_samples(self):
        return self._target.n_samples_fit_

    @property
    def n_features(self):
        return self._target.n_features_in

    @property
    def input_shape(self):
        return (self.n_features,)

    @property
    def output_shape(self):
        return (1,)
```

<strong>El flujo automático:</strong>
1. El usuario carga un modelo de sklearn (ej: `KNeighborsClassifier`)
2. EmbedIA busca en `dict_layers` la tupla `(KNeighborsClassifier, SKLKnnWrapper)`
3. Usa `SKLKnnWrapper` para extraer propiedades siguiendo la interfaz de `DistanceBaseWrapper`
4. Pasa esas propiedades a la implementación en `layers/` para generar código C

<strong>Importante:</strong> Las propiedades definidas en la interfaz base son exactamente las que usarás en `function_implementation` de la carpeta `layers` para generar el código C.


### Carpeta Layers 📁 <A NAME="layers"></A>
En esta carpeta residen todas las implementaciones específicas de los algoritmos. La estructura está organizada en carpetas por tipo de algoritmo.

<strong>Estructura de carpetas:</strong>
```
layers/
  ├── dense/
  ├── convolution/
  ├── activation/
  ├── knn/
  │   ├── k_neighbors_classifier.py
  │   ├── k_neighbors_regressor.py
  │   └── __init__.py
  ├── svm/
  ├── decision_tree/
  └── ... (otros algoritmos)
```

<strong>Implementación de tu algoritmo:</strong>
Dentro de la carpeta correspondiente al algoritmo (ej: 📁`knn` para KNeighborsClassifier), creas uno o varios archivos Python con el nombre descriptivo del algoritmo (ej: `k_neighbors_classifier.py`).

En este archivo defines una clase que hereda de la clase base del algoritmo definida en `core` (ej: `KnnBaseLayer`). Las dos propiedades principales que debes implementar son:

1. **`function_implementation`**: Genera el string de C con la función de inicialización usando los datos extraídos del wrapper
2. **`invoke`**: Retorna la invocación a la función de predicción C

Ejemplo para KNeighborsClassifier:

```python
class KNeighborsClassifier(KnnBaseLayer):
    @property
    def function_implementation(self):
        name = self.name
        struct_type = self.struct_data_type

        # Generar código C para inicialización
        init_knn_layer = f'''
{struct_type} init_{name}_data(void){{
    uint16_t n_neighbors = {self.wrapper.n_neighbors};
    uint32_t n_rows = {self.wrapper.n_samples};
    uint16_t n_features = {self.wrapper.n_features};
'''
        # Formatear datos de entrenamiento
        features_data = "\\n".join(["{" + ", ".join(map(str, row)) + "}," for row in self.wrapper.fit_x[:-1]])
        labels_data = ','.join([str(y) for y in self.wrapper.y])

        init_knn_layer += f'''
    static float neighbors_features[][{self.wrapper.n_features}] = {features_data};
    static float neighbors_labels[] = {labels_data};

    k_neighbors_classifier_layer_t layer = {{ n_neighbors, n_rows, n_features, *neighbors_features, neighbors_labels}};
    return layer;
}}
'''
        return init_knn_layer

    def invoke(self, input_name, output_name):
        return f'''k_neighbors_classifier_layer({self.name}_data, {input_name}, &{output_name});'''
```

<strong>Acceso a propiedades del modelo:</strong>
En `function_implementation` tienes acceso a:
- `self.wrapper`: El wrapper creado en la sección anterior, con todas las propiedades del modelo entrenado
- `self.name`: El nombre de la capa
- `self.struct_data_type`: El tipo de datos de la estructura C

<strong>Nota sobre nombres:</strong>
La convención en EmbedIA es usar nombres descriptivos pero concisos. Por ejemplo: `k_neighbors_classifier`, `decision_tree_classifier`, etc.

### Registro en layers_implemented.py <A NAME="registration"></A>
El archivo `core/layers_implemented.py` es la pieza central que conecta todo automáticamente. Aquí defines un diccionario `dict_layers` que mapea clases de modelos (scikit-learn, TensorFlow, etc.) a sus implementaciones en EmbedIA.

<strong>¿Qué sucede cuando cargas un modelo?</strong>
```
Usuario carga modelo sklearn
    ↓
EmbedIA identifica su clase: sklearn.neighbors.KNeighborsClassifier
    ↓
Busca en dict_layers: KNeighborsClassifier → (KNeighborsClassifier_layer, SKLKnnWrapper)
    ↓
Instancia SKLKnnWrapper sobre el modelo
    ↓
Extrae propiedades (n_neighbors, n_features, etc.) siguiendo la interfaz DistanceBaseWrapper
    ↓
Pasa al código de layers/ para generar C
```

<strong>Estructura de dict_layers:</strong>
```python
dict_layers = {
    # Clave: Clase del modelo de la librería
    # Valor: Tupla (clase_implementación_en_layers, clase_wrapper)

    neighbors.KNeighborsClassifier: (KNeighborsClassifier, SKLKnnWrapper),
    neighbors.KNeighborsRegressor: (KNeighborsRegressor, SKLKnnWrapper),
    svm.SVC: (SVMClassifier, SKLSVMWrapper),
    tree.DecisionTreeClassifier: (DecisionTreeClassifier, SKLTreeWrapper),
    # ... más algoritmos
}
```

<strong>Implementación en layers_implemented.py:</strong>

```python
# Importar la clase del modelo de sklearn
from sklearn import neighbors, svm, tree

# Importar las implementaciones de capas
from embedia.layers.knn import KNeighborsClassifier, KNeighborsRegressor
from embedia.layers.svm import SVMClassifier
from embedia.layers.decision_tree import DecisionTreeClassifier

# Importar los wrappers de sklearn
from embedia.wrappers.sklearn import SKLKnnWrapper, SKLSVMWrapper, SKLTreeWrapper

# ... más imports ...

# Crear el diccionario
dict_layers = {
    # Algoritmos basados en distancia (KNN)
    neighbors.KNeighborsClassifier: (KNeighborsClassifier, SKLKnnWrapper),
    neighbors.KNeighborsRegressor: (KNeighborsRegressor, SKLKnnWrapper),

    # Algoritmos de máquinas de soporte vectorial (SVM)
    svm.SVC: (SVMClassifier, SKLSVMWrapper),
    svm.SVR: (SVMRegressor, SKLSVMWrapper),

    # Algoritmos basados en árboles
    tree.DecisionTreeClassifier: (DecisionTreeClassifier, SKLTreeWrapper),
    tree.DecisionTreeRegressor: (DecisionTreeRegressor, SKLTreeWrapper),

    # ... más algoritmos
}
```

<strong>El flujo de generación automática:</strong>

1. **Identificación** (automática): Cuando EmbedIA recibe un modelo, identifica su tipo
2. **Búsqueda** (automática): Busca en `dict_layers` la entrada correspondiente
3. **Instanciación** (automática): Crea una instancia del wrapper sobre el modelo
4. **Extracción** (automática): El wrapper extrae propiedades del modelo
5. **Generación** (automática): La clase de layers genera el código C usando esas propiedades

<strong>Claves importantes:</strong>
- ✅ **Único lugar para registrar**: Cuando quieres añadir un nuevo algoritmo, solo necesitas agregar una línea a `dict_layers`
- ✅ **Búsqueda por tipo**: El sistema usa `type()` del modelo para buscar la tupla correcta
- ✅ **Interfaz consistente**: Todos los wrappers del mismo tipo (ej: sklearn) heredan de la misma interfaz
- ⚠️ **Debe coincidir exactamente**: La clave debe ser la clase EXACTA del modelo (ej: `sklearn.neighbors.KNeighborsClassifier`, no una cadena)

Si falta un registro, verás:
```
KeyError: <class 'sklearn.neighbors.KNeighborsClassifier'> is not in dict_layers
```

Solución: Agrega la línea correspondiente en `dict_layers`.

## Usando EmbedIA 🤖 <A NAME="usingembedia"></A>

Una vez que has integrado tu algoritmo en EmbedIA, usarlo es simple. El sistema es completamente **automático** tras las escenas.

<strong>Pasos que realizas (usuario):</strong>
1. Entrenar un modelo de scikit-learn
2. Guardar el modelo
3. Cargar el modelo en EmbedIA
4. Llamar a `generate_embedia_model()`

<strong>Lo que EmbedIA hace automáticamente:</strong>
1. Identifica la clase del modelo (ej: `sklearn.neighbors.KNeighborsClassifier`)
2. Busca en `dict_layers` para encontrar:
   - ✅ El wrapper correcto (`SKLKnnWrapper`)
   - ✅ La clase de implementación correcta (`KNeighborsClassifier` en layers/knn/)
3. Instancia el wrapper sobre tu modelo
4. Extrae todas las propiedades necesarias (n_neighbors, n_features, datos de entrenamiento, etc.)
5. Genera el código C usando esas propiedades
6. Crea un proyecto listo para compilar

<strong>Código de usuario (muy simple):</strong>

```python
import joblib
from embedia import generate_embedia_model

# 1. Entrenar el modelo
from sklearn.neighbors import KNeighborsClassifier
model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train, y_train)

# 2. Guardar el modelo
joblib.dump(model, 'models/my_knn.pkl')

# 3. Cargar para conversión
model = joblib.load('models/my_knn.pkl')

# 4. Convertir a EmbedIA (EL SISTEMA HACE TODO AUTOMÁTICAMENTE)
generate_embedia_model(
    model,
    model_name="my_knn_model",
    target_mcu="esp32"  # o "generic"
)
```

<strong>¿Cómo sabe EmbedIA qué hacer?</strong>

Detrás de escenas sucede esto:

```python
# En core/layers_implemented.py existe:
dict_layers = {
    neighbors.KNeighborsClassifier: (KNeighborsClassifier, SKLKnnWrapper)
    # ↑ Clave del diccionario
                                      ↑ clase_implementación
                                                        ↑ wrapper
}

# Cuando procesa el modelo:
model_type = type(model)  # sklearn.neighbors.KNeighborsClassifier

# Busca en dict_layers
layer_class, wrapper_class = dict_layers[model_type]

# Instancia el wrapper
wrapper = wrapper_class(model)

# Extrae propiedades automáticamente
n_neighbors = wrapper.n_neighbors  # ← Define en SKLKnnWrapper
n_features = wrapper.n_features    # ← Define en SKLKnnWrapper
fit_x = wrapper.fit_x              # ← Define en SKLKnnWrapper

# Genera el codigo C
layer = layer_class(embedia_model, wrapper)
c_code = layer.function_implementation  # ← usa las propiedades
```

<strong>Resultado esperado:</strong>

Si todo salió bien deberíamos recibir un resultado como este:

```
+---------------------+-------------------+------------+--------+------+------------+
| EmbedIA Layer       | Name              | #Param(NT) | Shape  | MACs | Size (KiB) |
+---------------------+-------------------+------------+--------+------+------------+
| KNeighborsClassifier| s_k_l_knn_wrapper |          0 | (353,) |    0 |     0.000  |
+---------------------+-------------------+------------+--------+------+------------+
Total params (NT)....: 0
Total size in KiB....: 0.000
Total MACs operations: 0

Project knn_project exported in outputs/
```

<strong>Estructura del proyecto generado:</strong>

```
outputs/
└── knn_project/
    ├── main.c                 ← Programa principal
    ├── model_data.c           ← Datos del modelo (generados automáticamente)
    ├── model_data.h
    ├── predict.c              ← Función de predicción
    ├── predict.h
    ├── knn.c / knn.h          ← Código de KNN
    ├── distances.c / distances.h
    ├── CMakeLists.txt         ← Para compilación
    └── ... (configuración del proyecto)
```

En caso de no aparecer así, a continuación se listan algunos posibles errores y cómo resolverlos:

### Posibles errores⚠️ <A NAME="errors"></A>

**Error: `KeyError: <class 'sklearn.neighbors.KNeighborsClassifier'> is not in dict_layers`**
- **Causa**: Tu algoritmo no está registrado en `core/layers_implemented.py`
- **Solución**: Agrega la línea correspondiente a `dict_layers`
  ```python
  neighbors.KNeighborsClassifier: (KNeighborsClassifier, SKLKnnWrapper)
  ```

**Error: `AttributeError: 'SKL[nombre]Wrapper' object has no attribute '[propiedad]'`**
- **Causa**: Una propiedad necesaria no está definida en el wrapper
- **Solución**: Asegúrate que todas las propiedades usadas en `function_implementation` estén en el wrapper:
  ```python
  @property
  def n_neighbors(self):
      return self._target.n_neighbors
  ```

**Error: `AttributeError: module 'keras._tf_keras.keras.layers' has no attribute 'LocallyConnected1D'`**
- **Causa**: Versión incompatible de TensorFlow (problema de redes neuronales)
- **Solución**: Baja la versión de TensorFlow (ej: a 2.14)

**Error: `ImportError: No module named 'embedia.wrappers.sklearn'`**
- **Causa**: Imports incorrectos en `layers_implemented.py`
- **Solución**: Verifica la ruta de importación:
  ```python
  from embedia.wrappers.sklearn import SKLKnnWrapper  # Correcto
  from embedia.wrappers import SKLKnnWrapper  # INCORRECTO (no está en raíz)
  ```

**Error: `ImportError: No module named 'embedia.layers.knn'`**
- **Causa**: La carpeta o archivo no existe en `layers/`
- **Solución**: Verifica que existe:
  ```
  embedia/layers/knn/k_neighbors_classifier.py
                    ↑ carpeta
                                  ↑ dentro de este archivo está la clase KNeighborsClassifier
  ```

**Error: Errores en compilación C (syntax error, undeclared variable, etc.)**
- **Causa**: El código C generado tiene problemas
- **Solución**:
  1. Revisa `function_implementation` en tu clase de layers - verifica tipos de datos
  2. Asegúrate que `EmbediaFile` está incluyendo correctamente los archivos necesarios
  3. Verifica que los tipos de datos coincidan con `common.h`
  4. Mira el archivo `model_data.c` generado para ver el código C actual

<strong>Checklist de debugging:</strong>
Si EmbedIA no reconoce tu algoritmo:

- [ ] ¿Existe el archivo base en `core/[nombre]_base_layer.py`?
- [ ] ¿Existe la carpeta `layers/[nombre del algoritmo]/`?
- [ ] ¿Existe el archivo de implementación en `layers/[nombre]/[nombre_descriptivo].py`?
- [ ] ¿Existe el wrapper en `wrappers/[biblioteca]/...py`?
- [ ] ¿Está el registro en `core/layers_implemented.py`?
- [ ] ¿Los imports en `layers_implemented.py` son correctos?
- [ ] ¿La clase del modelo en dict_layers es EXACTA? (no una cadena, sino la clase real)
- [ ] ¿El wrapper implementa TODAS las propiedades necesarias?
- [ ] ¿Los nombres de archivo (.h y .c) existen en `libraries/mcu/generic/[datatype]/`?

<strong>Felicidades 🎉, ¡integraste un nuevo algoritmo en EmbedIA!</strong>

Para poder descargar todos los archivos que se generaron, puedes usarse estas líneas de código:
```python
from google.colab import files

!zip -r embedia_project.zip 'outputs/knn_project'
files.download('/content/EmbedIA/embedia_project.zip')
```

<small>Nota: Es posible que en main.c se haya generado un `#include "neural_net.h"` no deseado. Si es así, simplemente elimina esa línea.</small>

### 🙏 Muchas gracias por llegar al final de la guía, ¡espero que te haya sido útil! 😄👋

---

## Resumen de la Arquitectura Mejorada

Esta guía documenta la arquitectura de EmbedIA v0.96.0+, con las siguientes mejoras principales:

### 1. **Sistema Modular de MCU** 🏗️
- `libraries/mcu/generic/`: Implementaciones base, portables
- `libraries/mcu/[plataforma]/`: Optimizaciones específicas (esp32, etc.)
- `libraries/mcu/template/`: Archivos plantilla reutilizables
- Búsqueda automática: plataforma → genérico → plantilla

### 2. **Separación de Interfaz e Implementación** 🎯
- **Interfaces base** (raíz de wrappers/): Define QUÉ necesita un algoritmo
- **Implementaciones** (carpetas por librería): Implementan CÓMO extraer propiedades
  - `sklearn/`: Para scikit-learn
  - `tensorflow/`: Para TensorFlow/Keras
  - `larq/`: Para Larq

### 3. **EmbediaFile para Control de Includes** 📄
- Especifica qué archivos incluir y dónde buscarlos
- Permite inyectar defines para código condicional (#ifdef)
- Ejemplo: `EmbediaFile('knn.h', defines=['OPTIMIZED_ESP32'])`

### 4. **Automatización Completa** 🤖
- `dict_layers` conecta modelos con sus implementaciones
- El sistema automáticamente:
  1. Identifica el tipo de modelo
  2. Busca en dict_layers
  3. Instancia el wrapper
  4. Extrae propiedades
  5. Genera código C

### 5. **Estructura Clara de Archivos** 📁
```
embedia/
├── core/                 ← Clases base de algoritmos
│   ├── [algoritmo]_base_layer.py
│   └── layers_implemented.py  ← dict_layers (pieza central)
├── wrappers/             ← Interfaces e implementaciones
│   ├── [interfaz]_base.py
│   ├── sklearn/
│   ├── tensorflow/
│   └── larq/
├── layers/               ← Generadores de código específicos
│   ├── [algoritmo]/
│   │   └── [implementacion].py
│   └── ...
└── libraries/
    └── mcu/
        ├── generic/      ← Implementaciones portables
        ├── esp32/        ← Optimizaciones ESP32
        └── template/     ← Plantillas base
```

---

**Documentación actualizada**: Mayo 2026
**Versión de EmbedIA**: 0.96.0+
**Status**: ✅ Actualizada con nueva arquitectura
